## Calculate marker gene scores

Load library

In [1]:
# Path-related libraries
import os
from pyhere import here  # For reproducible relative paths
import sys # system specific parameters
from pathlib import Path # File system paths

# AnnData and single-cell analysis libraries
import scanpy as sc       # For preprocessing and visualization of single-cell data
import anndata as ad      # For handling AnnData objects

# Numerical operations
import numpy as np        # For numerical computations and array manipulations
import pandas as pd

# Data visualization
import seaborn as sns        # Statistical data visualization
import matplotlib.pyplot as plt  # Plotting interface
import matplotlib            # Base matplotlib functionality
from matplotlib.backends.backend_pdf import PdfPages  # Save plots to multi-page PDFs

# Parallize processes
from joblib import Parallel, delayed


# Download from internet
import requests # http

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import my_anndata as ma                    # Custom AnnData utilities
import misc as misc   

Parameters

In [2]:
# save paths
base_dir = str(here('data/integrate/second_pass/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
objects_dir = os.path.join(base_dir, 'objects') 

anndata_dir = str(here('data/anndata/'))
harmo_dir = Path(here('data/marker_database/harmonized'))

/work/islet_cartography_scrna/data/marker_database Directory already exists!


In [3]:
# Plotting
plt.style.use('default') 

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
 # 
SMALL_SIZE = 4
MEDIUM_SIZE = 6
BIGGER_SIZE = 8
 
plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=SMALL_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

sc.set_figure_params(figsize=(2, 2), frameon=False, dpi_save= 300)

sc.settings.figdir = plot_dir
%config InlineBackend.print_figure_kwargs={'facecolor': 'w'}
%config InlineBackend.figure_format='retina'

Pancreatic cells marker genes lists

Import data

In [9]:
# AnnData object
adata = ad.read_h5ad(os.path.join(anndata_dir, 'AD_combined.h5ad'))

# Marker genes
azimuth_genes = pd.read_csv(os.path.join(harmo_dir, 'azimuth_genes.csv'))
azimuth_genes = azimuth_genes.groupby("cell_type")["gene"].apply(list).to_dict()

# Make directory of directies of genes 
marker_genes_dict = {'azimuth' : azimuth_genes}

In [16]:
# Combine both databases into one iterable
tasks = []
for database, genesets in marker_genes_dict.items():
    for celltype, genelist in genesets.items():
        score_name = f"{database}_{celltype}_score"
        tasks.append((score_name, genelist))

# Define function to calculate score
def score_one(task):
    score_name, genelist = task
    tmp = sc.tl.score_genes(
        adata=adata,
        gene_list=genelist,
        copy=True,
        score_name=score_name,
        use_raw=False,
    )
    return tmp.obs[[score_name]]

# run in parallel 
score_df = Parallel(n_jobs=3)(
    delayed(score_one)(task) for task in tasks
)

# combine results
score_df = pd.concat(score_df, axis=1)

# ensure same index order as adata
score_df = score_df.loc[adata.obs_names]

# Save file
score_df.to_csv(os.path.join(files_dir, 'marker_gene_scores.csv'))

In [17]:
# markers = pd.read_csv(os.path.join(files_dir, 'marker_gene_scores.csv'), index_col= 0)